In [ ]:
# Universal Setup

!pip install -q faiss-cpu chromadb qdrant-client rank-bm25 \
    sentence-transformers psutil pymupdf tiktoken groq google-genai \
    pandas matplotlib seaborn scikit-learn

import os, re, json, time, gc, random, glob
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

BASE    = '/content/drive/MyDrive/rag_term_paper'
PDF_DIR = f'{BASE}/data/raw_pdfs'
PROC    = f'{BASE}/data/processed'
EMB     = f'{BASE}/data/embeddings'
RES     = f'{BASE}/results'
for d in (PDF_DIR, PROC, EMB, RES):
    os.makedirs(d, exist_ok=True)

EMBED_MODEL  = 'BAAI/bge-small-en-v1.5'
BGE_PREFIX   = 'Represent this sentence for searching relevant passages: '
CHUNK_TOKENS = 500
OVERLAP      = 50
print("Setup complete. Base directory:", BASE)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
ibis-framework 9.5.0 r

In [ ]:
# Download 10 Papers & Save Manifest
import urllib.request

papers = {
    "doc01_rag_original":      "https://arxiv.org/pdf/2005.11401.pdf",
    "doc02_rag_survey":        "https://arxiv.org/pdf/2312.10997.pdf",
    "doc03_dpr":               "https://arxiv.org/pdf/2004.04906.pdf",
    "doc04_ragas":             "https://arxiv.org/pdf/2309.15217.pdf",
    "doc05_self_rag":          "https://arxiv.org/pdf/2310.11511.pdf",
    "doc06_transformer":       "https://arxiv.org/pdf/1706.03762.pdf",
    "doc07_lora":              "https://arxiv.org/pdf/2106.09685.pdf",
    "doc08_flash_attention":   "https://arxiv.org/pdf/2205.14135.pdf",
    "doc09_rag_survey_huang":  "https://arxiv.org/pdf/2404.10981.pdf",
    "doc10_comprehensive_rag": "https://arxiv.org/pdf/2410.12837.pdf",
}

opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'Mozilla/5.0 (Research Benchmark Script)')]
urllib.request.install_opener(opener)

for doc_id, url in papers.items():
    dest = os.path.join(PDF_DIR, f"{doc_id}.pdf")
    if not os.path.exists(dest):
        print(f"Downloading {doc_id}...")
        urllib.request.urlretrieve(url, dest)
        time.sleep(1)
    else:
        print(f"Skipping {doc_id} (exists)")

manifest = pd.DataFrame([
    {"doc_id": k, "url": v, "filename": f"{k}.pdf"}
    for k, v in papers.items()
])
manifest.to_csv(f'{PROC}/corpus_manifest.csv', index=False)
print(f"\nSaved corpus_manifest.csv ({len(papers)} PDFs in {PDF_DIR})")


Saved corpus_manifest.csv (10 PDFs in /content/drive/MyDrive/rag_term_paper/data/raw_pdfs)


In [ ]:
# Chunking with Bibliography Filter
import pymupdf, tiktoken
enc = tiktoken.get_encoding('cl100k_base')

def extract_text(path):
    with pymupdf.open(path) as doc:
        return '\n'.join(page.get_text() for page in doc)

def looks_like_references(t):
    hits = len(re.findall(
        r'arXiv|Proceedings|Conference|Journal|et al\.|'
        r'\b(?:19|20)\d\d\b', t))
    return hits / max(len(t), 1) * 1000 > 6

def chunk_doc(text, doc_id, size=CHUNK_TOKENS, overlap=OVERLAP):
    toks = enc.encode(text)
    out, start, step = [], 0, size - overlap
    while start < len(toks):
        piece = enc.decode(toks[start:start + size]).strip()
        if len(piece) > 40 and not looks_like_references(piece):
            out.append({
                'chunk_id': f'{doc_id}_chunk_{len(out):04d}',
                'doc_id': doc_id,
                'token_count': min(size, len(toks) - start),
                'text': piece
            })
        if start + size >= len(toks):
            break
        start += step
    return out

all_chunks = []
for path in sorted(glob.glob(f'{PDF_DIR}/*.pdf')):
    doc_id = os.path.splitext(os.path.basename(path))[0]
    ch = chunk_doc(extract_text(path), doc_id)
    all_chunks += ch
    print(f"{doc_id}: {len(ch)} chunks")

with open(f'{PROC}/chunks.json', 'w') as f:
    json.dump(all_chunks, f, indent=1)
print(f'\nTotal chunks saved: {len(all_chunks)}')

doc01_rag_original: 29 chunks
doc02_rag_survey: 37 chunks
doc03_dpr: 26 chunks
doc04_ragas: 13 chunks
doc05_self_rag: 40 chunks
doc06_transformer: 17 chunks
doc07_lora: 40 chunks
doc08_flash_attention: 74 chunks
doc09_rag_survey_huang: 50 chunks
doc10_comprehensive_rag: 19 chunks

Total chunks saved: 345


In [ ]:
# 20 Evaluation Queries
eval_queries = [
    # Conceptual (q01-q08) — dense search targets
    {"query_id": "q01", "category": "conceptual", "answerable": True,
     "query": "How does Retrieval-Augmented Generation combine parametric and non-parametric memory?"},
    {"query_id": "q02", "category": "conceptual", "answerable": True,
     "query": "What are the core differences between Naive RAG, Advanced RAG, and Modular RAG architectures?"},
    {"query_id": "q03", "category": "conceptual", "answerable": True,
     "query": "How does Dense Passage Retrieval outperform traditional sparse BM25 models on open-domain QA?"},
    {"query_id": "q04", "category": "conceptual", "answerable": True,
     "query": "Explain how Self-RAG utilizes reflection tokens to determine when retrieval is necessary."},
    {"query_id": "q05", "category": "conceptual", "answerable": True,
     "query": "What is the primary mechanism behind Multi-Head Attention in the standard Transformer architecture?"},
    {"query_id": "q06", "category": "conceptual", "answerable": True,
     "query": "Why does Low-Rank Adaptation (LoRA) significantly reduce the number of trainable parameters during fine-tuning?"},
    {"query_id": "q07", "category": "conceptual", "answerable": True,
     "query": "How does FlashAttention optimize IO memory access between GPU HBM and SRAM to accelerate computation?"},
    {"query_id": "q08", "category": "conceptual", "answerable": True,
     "query": "What are the key failure modes and hallucination risks identified across modern RAG survey papers?"},
    # Keyword/Metric (q09-q15) — BM25 targets
    {"query_id": "q09", "category": "keyword_metric", "answerable": True,
     "query": "What exact BLEU score did the original Transformer model achieve on the WMT 2014 English-to-German translation benchmark?"},
    {"query_id": "q10", "category": "keyword_metric", "answerable": True,
     "query": "What is the rank r used in the default LoRA configuration for GPT-3 175B fine-tuning experiments?"},
    {"query_id": "q11", "category": "keyword_metric", "answerable": True,
     "query": "How many trainable parameters does LoRA use compared to full fine-tuning of GPT-3?"},
    {"query_id": "q12", "category": "keyword_metric", "answerable": True,
     "query": "What datasets were used in the original RAG paper to evaluate its sequence-to-sequence performance?"},
    {"query_id": "q13", "category": "keyword_metric", "answerable": True,
     "query": "What block sizes Bc and Br does the FlashAttention algorithm use for tiling, and how are they computed from the SRAM size M?"},
    {"query_id": "q14", "category": "keyword_metric", "answerable": True,
     "query": "What is the Exact Match score reported by Dense Passage Retrieval on Natural Questions?"},
    {"query_id": "q15", "category": "keyword_metric", "answerable": True,
     "query": "What specific evaluation metrics does the RAGAS framework define for automated RAG assessment?"},
    # Out-of-scope (q16-q20) — unanswerable
    {"query_id": "q16", "category": "out_of_scope", "answerable": False,
     "query": "What is the current stock price of NVIDIA and how has it changed in the last quarter?"},
    {"query_id": "q17", "category": "out_of_scope", "answerable": False,
     "query": "Explain the rules and scoring system of international cricket."},
    {"query_id": "q18", "category": "out_of_scope", "answerable": False,
     "query": "What are the health benefits of a Mediterranean diet according to recent clinical trials?"},
    {"query_id": "q19", "category": "out_of_scope", "answerable": False,
     "query": "How does photosynthesis convert carbon dioxide and water into glucose and oxygen?"},
    {"query_id": "q20", "category": "out_of_scope", "answerable": False,
     "query": "What programming language is best for building mobile applications in 2025?"},
]

with open(f'{PROC}/eval_queries.json', 'w') as f:
    json.dump(eval_queries, f, indent=2)
print(f"Saved {len(eval_queries)} queries to {PROC}/eval_queries.json")

Saved 20 queries to /content/drive/MyDrive/rag_term_paper/data/processed/eval_queries.json


In [ ]:
# Embed + Truncation Check + Freeze
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(EMBED_MODEL, device=device)
print(f"Model: {EMBED_MODEL} | Device: {device}")

chunks = json.load(open(f'{PROC}/chunks.json'))
queries = json.load(open(f'{PROC}/eval_queries.json'))

# Truncation check
lens = [len(model.tokenizer(c['text'])['input_ids']) for c in chunks]
over = sum(l > model.max_seq_length for l in lens)
print(f"Max seq length: {model.max_seq_length} | Longest chunk: {max(lens)} "
      f"| Truncated: {over} ({over/len(lens):.1%})")
if over / len(lens) > 0.05:
    print("⚠️  > 5% truncated! Consider setting CHUNK_TOKENS = 380 in Cell 0 and re-running Cell 2.")

# Embed with SAME prefix for ALL queries
chunk_texts = [c['text'] for c in chunks]
query_texts = [BGE_PREFIX + q['query'] for q in queries]

print(f"Encoding {len(chunk_texts)} chunks...")
doc_embeddings = model.encode(chunk_texts, batch_size=32,
                              normalize_embeddings=True, show_progress_bar=True)

print(f"Encoding {len(query_texts)} queries...")
query_embeddings = model.encode(query_texts, batch_size=32,
                                normalize_embeddings=True)

chunk_ids = [c['chunk_id'] for c in chunks]
np.save(f'{EMB}/doc_embeddings.npy', doc_embeddings)
np.save(f'{EMB}/query_embeddings.npy', query_embeddings)
with open(f'{EMB}/chunk_ids.json', 'w') as f:
    json.dump(chunk_ids, f)
print(f"\nFrozen: doc={doc_embeddings.shape}, query={query_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

... (7 outputs left out)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (528 > 512). Running this sequence through the model will result in indexing errors


Model: BAAI/bge-small-en-v1.5 | Device: cuda
Max seq length: 512 | Longest chunk: 572 | Truncated: 19 (5.5%)
⚠️  > 5% truncated! Consider setting CHUNK_TOKENS = 380 in Cell 0 and re-running Cell 2.
Encoding 345 chunks...


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Encoding 20 queries...

Frozen: doc=(345, 384), query=(20, 384)


In [ ]:
# Brute-Force NumPy Exact Baseline
sim_matrix = np.dot(query_embeddings, doc_embeddings.T)
query_ids = [q['query_id'] for q in queries]

exact_topk = {}
for q_idx, q_id in enumerate(query_ids):
    sorted_idx = np.argsort(sim_matrix[q_idx])[::-1][:20]
    exact_topk[q_id] = {
        "query": queries[q_idx]["query"],
        "category": queries[q_idx]["category"],
        "answerable": queries[q_idx]["answerable"],
        "top_chunks": [
            {"rank": r+1, "chunk_id": chunk_ids[i],
             "score": float(sim_matrix[q_idx][i]),
             "text_preview": chunks[i]["text"][:160]}
            for r, i in enumerate(sorted_idx)
        ]
    }

with open(f'{PROC}/numpy_exact_topk.json', 'w') as f:
    json.dump(exact_topk, f, indent=2)
print(f"Exact-search reference saved ({len(exact_topk)} queries)")

# Sanity check
for qid in ["q01", "q16"]:
    t = exact_topk[qid]
    print(f"\n{qid} [{t['category']}]: top-1 score = "
          f"{t['top_chunks'][0]['score']:.4f} → {t['top_chunks'][0]['chunk_id']}")

Exact-search reference saved (20 queries)

q01 [conceptual]: top-1 score = 0.8452 → doc01_rag_original_chunk_0003

q16 [out_of_scope]: top-1 score = 0.6243 → doc08_flash_attention_chunk_0053
